# 2. Install and initialize

### Cell 1 — Dependencies

In [ ]:
!pip -q install pandas numpy requests \
scikit-learn matplotlib seaborn \
shap scipy statsmodels gseapy \
anndata scanpy

### Cell 2 — Project folders

In [ ]:
from pathlib import Path
import os
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE = Path("/content/ASTRABIO_02")

FOLDERS = [
    "raw",
    "processed",
    "metadata",
    "models",
    "results",
    "figures"
]

for folder in FOLDERS:
    (BASE / folder).mkdir(
        parents=True, exist_ok=True
    )

print("ASTRABIO ML environment ready")

# 3. NASA human dataset discovery

### Cell 3 — Configure datasets

In [ ]:
ACCESSIONS = [
    "OSD-569",
    "OSD-570",
    "OSD-571",
    "OSD-574",
    "OSD-575"
]

def accession_number(acc):
    return acc.replace("OSD-", "")

print(ACCESSIONS)

### Cell 4 — Download metadata

In [ ]:
def get_metadata(accession):

    number = accession_number(accession)

    url = (
        "https://osdr.nasa.gov/osdr/data/"
        f"osd/meta/{number}"
    )

    response = requests.get(
        url, timeout=90
    )

    response.raise_for_status()

    return response.json()


metadata = {}

for accession in ACCESSIONS:

    try:
        data = get_metadata(accession)
        metadata[accession] = data

        path = (
            BASE / "metadata" /
            f"{accession}_metadata.json"
        )

        path.write_text(
            json.dumps(data, indent=2)
        )

        print("SUCCESS:", accession)

    except Exception as e:
        print("ERROR:", accession, e)

### Cell 5 — Inspect study metadata

In [ ]:
def inspect_metadata(data):

    study = data.get("study", {})

    for study_id, details in study.items():

        print("\nSTUDY:", study_id)

        print(
            json.dumps(
                details,
                indent=2
            )[:5000]
        )


for accession, data in metadata.items():

    print("\n", "=" * 50)
    inspect_metadata(data)

# 4. Retrieve NASA biological data file manifests

### Cell 6 — List available files

In [ ]:
def get_file_manifest(accession):

    number = accession_number(accession)

    url = (
        "https://osdr.nasa.gov/osdr/data/"
        f"osd/files/{number}"
    )

    response = requests.get(
        url, timeout=90
    )

    response.raise_for_status()

    return response.json()


all_files = []

for accession in ACCESSIONS:

    try:

        manifest = get_file_manifest(accession)

        studies = manifest.get("studies", {})

        for study_id, study_info in studies.items():

            for f in study_info.get(
                "study_files", []
            ):

                all_files.append({
                    "accession": accession,
                    "study_id": study_id,
                    "filename": f.get("file_name"),
                    "category": f.get("category"),
                    "size_bytes": f.get("file_size"),
                    "restricted": f.get("restricted"),
                    "remote_url": f.get("remote_url")
                })

    except Exception as e:
        print("Manifest error:", accession, e)


files_df = pd.DataFrame(all_files)

display(files_df)

### Cell 7 — Find suitable processed biological files

In [ ]:
if not files_df.empty:

    files_df["size_mb"] = (
        pd.to_numeric(
            files_df["size_bytes"],
            errors="coerce"
        ) / (1024 ** 2)
    )

    files_df["is_public"] = (
        files_df["restricted"]
        .astype(str)
        .str.lower()
        .eq("false")
    )

    candidates = files_df[
        files_df["is_public"] &
        files_df["filename"].str.contains(
            r"\.(csv|tsv|txt)(\.gz)?$",
            case=False,
            na=False
        )
    ]

    display(
        candidates.sort_values(
            "size_mb"
        )
    )

### Cell 8 — Download a selected public file

In [ ]:
SELECTED_ACCESSION = "OSD-569"
SELECTED_FILENAME = ""

def download_osdr_file(
    accession,
    filename
):

    row = files_df[
        (files_df["accession"] == accession) &
        (files_df["filename"] == filename)
    ]

    if row.empty:
        raise ValueError("File not found")

    record = row.iloc[0]

    if not record["is_public"]:
        raise PermissionError(
            "This file is restricted."
        )

    size_mb = record["size_mb"]

    if pd.isna(size_mb) or size_mb > 500:
        raise ValueError(
            "File exceeds the 500 MB starter limit."
        )

    remote_url = record["remote_url"]

    if not remote_url:
        raise ValueError("No download URL available")

    url = "https://osdr.nasa.gov" + remote_url

    response = requests.get(
        url,
        stream=True,
        timeout=180
    )

    response.raise_for_status()

    output = (
        BASE / "raw" / filename
    )

    with open(output, "wb") as f:

        for chunk in response.iter_content(
            chunk_size=1024 * 1024
        ):

            if chunk:
                f.write(chunk)

    return output

# 5. Load and inspect biological measurements

### Cell 9 — Load processed expression data

In [ ]:
def load_expression_file(path):

    path = str(path)

    if path.endswith(".gz"):
        df = pd.read_csv(
            path,
            sep=None,
            engine="python",
            compression="gzip"
        )

    else:
        df = pd.read_csv(
            path,
            sep=None,
            engine="python"
        )

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    display(df.head())

    return df

### Cell 10 — Identify biological data structure

In [ ]:
def inspect_biological_data(df):

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("\nMissing values:")
    display(
        df.isna().sum().sort_values(
            ascending=False
        ).head(20)
    )

    print("\nData types:")
    display(df.dtypes)

    numeric = df.select_dtypes(
        include=np.number
    )

    print("\nNumeric measurements:", numeric.shape)

    display(numeric.describe().T.head(20))

    return numeric

In [ ]:
# expression_df = load_expression_file(downloaded_file)
# numeric_df = inspect_biological_data(expression_df)

### Cell 11 — Quality control

In [ ]:
def quality_control(df):

    print("Duplicate rows:", df.duplicated().sum())

    print("Missing percentage:")
    print(
        (df.isna().mean() * 100)
        .sort_values(ascending=False)
    )

    numeric = df.select_dtypes(
        include=np.number
    )

    print("\nInfinite values:")
    print(
        np.isinf(
            numeric.to_numpy()
        ).sum()
    )

    print("\nConstant columns:")
    print(
        numeric.columns[
            numeric.nunique() <= 1
        ].tolist()
    )

# 6. Exploratory biological analysis

### Cell 12 — Distribution visualization

In [ ]:
numeric = expression_df.select_dtypes(
    include=np.number
)

numeric.hist(
    figsize=(16, 12),
    bins=30
)

plt.suptitle(
    "ASTRABIO Biological Data Distribution"
)

plt.tight_layout()
plt.show()

### Cell 13 — Correlation analysis

In [ ]:
correlation = numeric.corr()

plt.figure(figsize=(12, 9))

sns.heatmap(
    correlation,
    cmap="coolwarm",
    center=0
)

plt.title(
    "Biological Measurement Correlations"
)

plt.show()

### Cell 14 — PCA exploration

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = numeric.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.dropna(axis=1, how="all")

X = X.fillna(
    X.median()
)

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)

components = pca.fit_transform(
    X_scaled
)

plt.figure(figsize=(9, 6))

plt.scatter(
    components[:, 0],
    components[:, 1],
    alpha=0.7
)

plt.xlabel("PC1")
plt.ylabel("PC2")

plt.title(
    "ASTRABIO Biological PCA Exploration"
)

plt.show()

print(
    "Explained variance:",
    pca.explained_variance_ratio_
)

# 7. Train the first ASTRABIO ML model

### Cell 15 — Prepare labeled training data

In [ ]:
TARGET = "phase"
GROUP = "subject_id"

FEATURES = [
    # Add verified numerical biological features
    # Example: "GENE_A", "GENE_B", "GENE_C"
]

### Cell 16 — Train/test separation

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score
)

def train_astrabio_model(df):

    required = FEATURES + [TARGET, GROUP]

    missing = [
        c for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing columns: {missing}"
        )

    data = df.dropna(
        subset=[TARGET, GROUP]
    ).copy()

    X = data[FEATURES].apply(
        pd.to_numeric,
        errors="coerce"
    )

    y = data[TARGET]
    groups = data[GROUP]

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.25,
        random_state=42
    )

    train_idx, test_idx = next(
        splitter.split(X, y, groups)
    )

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    if y_train.nunique() < 2:
        raise ValueError(
            "Training data has fewer than two classes."
        )

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced"
            )
        )
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )

    print(
        "Balanced accuracy:",
        balanced_accuracy_score(
            y_test,
            predictions
        )
    )

    print("Confusion matrix:")

    print(
        confusion_matrix(
            y_test,
            predictions
        )
    )

    return model, X_train, X_test, y_train, y_test

In [ ]:
# model, X_train, X_test, y_train, y_test = (
#     train_astrabio_model(expression_df)
# )

# 8. SHAP explainability

### Cell 17 — Explain model features

In [ ]:
import shap

def explain_astrabio_model(
    model,
    X_train,
    X_test
):

    classifier = model.named_steps[
        "classifier"
    ]

    imputer = model.named_steps[
        "imputer"
    ]

    scaler = model.named_steps[
        "scaler"
    ]

    train_processed = scaler.transform(
        imputer.transform(X_train)
    )

    test_processed = scaler.transform(
        imputer.transform(X_test)
    )

    explainer = shap.LinearExplainer(
        classifier,
        train_processed
    )

    values = explainer(
        test_processed
    )

    shap.summary_plot(
        values,
        test_processed,
        feature_names=FEATURES
    )

    return values

In [ ]:
# shap_values = explain_astrabio_model(
#     model, X_train, X_test
# )

# 9. Biological pathway analysis

### Cell 18 — Install and configure GSEApy

In [ ]:
import gseapy as gp

gene_list = [
    # "TP53",
    # "IL6",
    # "TNF"
]

### Cell 19 — Investigate biological pathways

In [ ]:
def run_pathway_analysis(genes):

    if not genes:
        raise ValueError(
            "Provide verified gene symbols first."
        )

    result = gp.enrichr(
        gene_list=genes,
        gene_sets=[
            "GO_Biological_Process_2023",
            "KEGG_2021_Human"
        ],
        organism="Human",
        outdir=str(
            BASE / "results" / "pathways"
        ),
        cutoff=0.05
    )

    display(result.results.head(20))

    return result.results

# 10. Export trained models for ASTRABIO Notebook 01

### Cell 20 — Save the model

In [ ]:
import joblib

MODEL_PATH = BASE / "models" / "astrabio_model.pkl"

# joblib.dump(model, MODEL_PATH)

print("Model export path:", MODEL_PATH)

### Cell 21 — Export model metadata

In [ ]:
model_info = {
    "project": "ASTRABIO",
    "model_type": "Logistic Regression",
    "training_source": "NASA OSDR",
    "target": TARGET,
    "features": FEATURES,
    "grouping_variable": GROUP,
    "purpose": "Experimental biological pattern classification",
    "clinical_validation": False
}

with open(
    BASE / "models" / "model_metadata.json",
    "w"
) as f:

    json.dump(
        model_info,
        f,
        indent=4
    )

### Cell 22 — Package project results

In [ ]:
import shutil

shutil.make_archive(
    "/content/ASTRABIO_02_RESULTS",
    "zip",
    BASE
)

from google.colab import files

files.download(
    "/content/ASTRABIO_02_RESULTS.zip"
)